In [ ]:
# @title
!pip install transformers
!pip install datasets
!pip install scikit-learn
!pip install pandas
!pip install matplotlib
!pip install seaborn
!pip install torch

In [ ]:
import pandas as pd

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = 1
true["label"] = 0
df = pd.concat([fake, true])
df["content"] = df["title"] + " " + df["text"]
df = df[["content","label"]]
df

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.isnull().sum()


In [ ]:
df['label'].value_counts()

In [ ]:
df.head(10)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='label', data=df)
plt.show()

In [ ]:
import torch
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["content"],
    df["label"],
    test_size=0.2,
    random_state=42
)

In [ ]:
print(len(train_texts))
print(len(val_texts))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding="max_length",
    max_length=256
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding="max_length",
    max_length=256
)

In [ ]:
print(train_encodings['input_ids'][0])

In [ ]:
class FakeNewsDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = FakeNewsDataset(train_encodings, train_labels.tolist())
val_dataset = FakeNewsDataset(val_encodings, val_labels.tolist())

In [ ]:
sample = train_dataset[0]

for key, value in sample.items():
    print(key, value.shape)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    learning_rate=3e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_dir="./logs",

    load_best_model_at_end=True,

    weight_decay=0.01
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
print(train_dataset[0])

In [ ]:
print(df["content"].iloc[0])

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("fake_news_model")
tokenizer.save_pretrained("fake_news_model")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

trainer.save_model("/content/drive/MyDrive/fake_news_model")
tokenizer.save_pretrained("/content/drive/MyDrive/fake_news_model")

In [ ]:
trainer.evaluate()

In [ ]:
predictions = trainer.predict(val_dataset)

In [ ]:
preds = predictions.predictions.argmax(axis=1)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(val_labels, preds))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(val_labels, preds)
print(cm)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
wrong_predictions = []

for text, true_label, pred_label in zip(val_texts, val_labels, preds):
    if true_label != pred_label:
        wrong_predictions.append((text, true_label, pred_label))

len(wrong_predictions)

In [ ]:
!pip install gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/fake_news_model")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/fake_news_model")

In [ ]:
model.eval()

In [ ]:
import torch

def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    pred = torch.argmax(probs).item()

    if pred == 0:
        return "Real News"
    else:
        return "Fake News"

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=5, placeholder="Enter news text here"),
    outputs="text",
    title="Fake News Detection using DistilBERT"
)

interface.launch(share=True)